- Write time optimization  -> Optimized Writes
- Post-write compaction    -> Auto Compaction


Write Time Optimization applied when a dataframe writes its data into table, all it does is automatic reshuffles/coalesces data before writing to produce better sized files

- Task1 -> 5 MB file
- Task2 -> 7 MB file
- Task3 -> 3 MB file
- Task4 -> 6 MB file
- ...

if write optimize enabled

- Task1+Task2 + ... -> 128 MB file
- Task3+Task4 + ... -> 128 MB file

In [0]:
DROP TABLE IF EXISTS  orderdb.brands

In [0]:
-- change table name if shared acc
CREATE TABLE IF NOT EXISTS orderdb.brands (
  brand_id INT,
  brand_name STRING,
  category STRING
)
USING DELTA;

org.apache.spark.sql.catalyst.analysis.TableAlreadyExistsException: [TABLE_OR_VIEW_ALREADY_EXISTS] Cannot create table or view `orderdb`.`brands` because it already exists.
Choose a different name, drop the existing object, add the IF NOT EXISTS clause to tolerate pre-existing objects, add the OR REPLACE clause to replace the existing materialized view, or add the OR REFRESH clause to refresh the existing streaming table. SQLSTATE: 42P07
	at org.apache.spark.sql.errors.QueryCompilationErrors$.tableAlreadyExistsError(QueryCompilationErrors.scala:2263)
	at org.apache.spark.sql.execution.datasources.v2.CreateTableExec.run(CreateTableExec.scala:87)
	at org.apache.spark.sql.execution.datasources.v2.V2CommandExec.$anonfun$result$2(V2CommandExec.scala:48)
	at org.apache.spark.sql.execution.SparkPlan.runCommandInAetherOrSpark(SparkPlan.scala:194)
	at org.apache.spark.sql.execution.datasources.v2.V2CommandExec.$anonfun$result$1(V2CommandExec.scala:48)
	at com.databricks.spark.util.FrameProfiler

In [0]:
SET spark.databricks.delta.optimizeWrite.enabled; -- actually get the property

In [0]:
SET spark.databricks.delta.autoCompact.enabled;

In [0]:
-- disable auto for demo purpose only
ALTER TABLE orderdb.brands
SET TBLPROPERTIES (
 delta.autoOptimize.optimizeWrite = false,
 delta.autoOptimize.autoCompact = false
);

In [0]:
INSERT INTO orderdb.brands VALUES (1,'Nike','Shoes');
INSERT INTO orderdb.brands VALUES (2,'Puma','Shoes');
INSERT INTO orderdb.brands VALUES (3,'Apple','Electronics');
-- repeat many times, just run same or random below cell

In [0]:
%python 
for i in range(100):
  spark.sql(f"""
  INSERT INTO orderdb.brands
  VALUES ({i}, 'Brand{i}', 'Demo')
  """)

In [0]:
-- small files, we did many inserts like 100 plus

DESCRIBE DETAIL orderdb.brands;
-- check for numFiles, sizeInBytes, numFiles are  real concern

In [0]:
DESCRIBE HISTORY orderdb.brands;

In [0]:
ALTER TABLE orderdb.brands
SET TBLPROPERTIES (
 delta.autoOptimize.optimizeWrite = true,
 delta.autoOptimize.autoCompact = true
);

-- delta.targetFileSize='256mb'   to control file size, else default 128mb
-- delta.tuneFileSizesForRewrites=true for merge heavy cdc workload

In [0]:
SET spark.databricks.delta.autoCompact.minNumFiles=20;
SET spark.databricks.delta.autoCompact.maxFileSize=268435456;

In [0]:
-- random data post optimize
INSERT INTO orderdb.brands
SELECT
id,
concat('Brand',id),
'Retail'
FROM range(10);

In [0]:
DESCRIBE DETAIL orderdb.brands;

In [0]:
DESCRIBE HISTORY orderdb.brands;
-- remember, we did manual optimize using like OPTIMIZE orderdb.brands; or OPTIMIZE orderdb.brands ZORDER BY (brand_id);
-- what happens is automatic

In [0]:
-- session level settings, even if a table is not marked for optimize, we can do at the session level. 
-- optimized writes
SET spark.databricks.delta.optimizeWrite.enabled=true;

-- auto compaction
SET spark.databricks.delta.autoCompact.enabled=auto;


In [0]:
-- retension per table 

ALTER TABLE orderdb.brands
SET TBLPROPERTIES (
 delta.deletedFileRetentionDuration='30 days'
);

-- deletedFileRetentionDuration is for deleted files, parquet files (remove actions)

-- delta.logRetentionDuration, this is for history, remmeber _delta_log directory, **.json files
-- delta.logRetentionDuration controls transaction-log history
-- in _delta_log (json commits + checkpoint parquet files)

In [0]:
SHOW TBLPROPERTIES orderdb.brands;

In [0]:
DESCRIBE DETAIL orderdb.brands;

In [0]:
VACUUM orderdb.brands;  -- clean data older than 30 days, which is defined in this table, otherwise default is 7 days

In [0]:
SET spark.databricks.delta.retentionDurationCheck.enabled